##Create Inventory Ledger

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import *

GOLD = "capretailnew.gold"
gold_fact_inventory = spark.table(GOLD + ".gold_fact_inventory_movements")

# Define window
inventory_window = (
    Window
    .partitionBy("product_id", "store_id")
    .orderBy("movement_ts", "movement_id")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

# Calculate running movement
gold_inventory_ledger = (
    gold_fact_inventory
    .withColumn(
        "movement_sequence",
        row_number().over(
            Window.partitionBy(
                "product_id",
                "store_id"
            ).orderBy(
                "movement_ts",
                "movement_id"
            )
        )
    )
    .withColumn(
        "cumulative_signed_movement",
        sum("quantity_change").over(
            inventory_window
        )
    )
)

# Save ledger
gold_inventory_ledger.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        GOLD + ".gold_inventory_movement_ledger"
    )

print("Inventory Movement Ledger created")

##Create Gold Daily Sales Summary

In [0]:
gold_fact_sales = spark.table(GOLD + ".gold_fact_sales")

gold_daily_sales = (
    gold_fact_sales
    .withColumn(
        "sales_date",
        to_date("order_ts")
    )
    .groupBy(
        "sales_date",
        "store_id",
        "store_name",
        "category"
    )
    .agg(
        countDistinct("order_id").alias("total_orders"),

        count("order_line_id").alias("total_order_lines"),

        sum("quantity").alias("total_quantity_sold"),

        sum("gross_sales").alias("gross_sales"),

        sum("discount_amount").alias("total_discount"),

        sum("net_sales").alias("net_sales")
    )
)

gold_daily_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        GOLD + ".gold_daily_sales_summary"
    )

print("Daily Sales Summary created")

##Create Store Sales KPIs

In [0]:
gold_store_kpis = (
    gold_fact_sales
    .groupBy(
        "store_id",
        "store_name",
        "city",
        "region"
    )
    .agg(
        countDistinct("order_id").alias("total_orders"),

        count("order_line_id").alias("total_order_lines"),

        sum("quantity").alias("total_quantity_sold"),

        sum("gross_sales").alias("gross_sales"),

        sum("discount_amount").alias("total_discount"),

        sum("net_sales").alias("total_net_sales")
    )
)

gold_store_kpis.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        GOLD + ".gold_store_sales_kpis"
    )

print("Store Sales KPIs created")

##Unit Of Measure Validation

In [0]:
gold_uom_check = (
    gold_fact_inventory
    .groupBy(
        "product_id",
        "product_name",
        "unit_of_measure"
    )
    .agg(
        count("*").alias("total_movements"),

        sum("quantity_change").alias(
            "total_quantity_change"
        )
    )
)

display(gold_uom_check)

In [0]:
display(
    gold_fact_inventory.filter(
        col("unit_of_measure").isNull()
    )
)

##Final Gold table validation

In [0]:
gold_tables = [
    "gold_dim_product",
    "gold_dim_store",
    "gold_dim_supplier",
    "gold_fact_sales",
    "gold_fact_returns",
    "gold_fact_inventory_movements",
    "gold_inventory_movement_ledger",
    "gold_daily_sales_summary",
    "gold_store_sales_kpis"
]

for table in gold_tables:

    df = spark.table(GOLD + "." + table)

    print(
        table,
        " | Records:",
        df.count()
    )